# Preprocesing & Analysis

In [ ]:
#Import data and anonymize
import pandas as pd
import numpy as np

df_part = pd.read_csv("/work/MaleneJensen#0692/exam/output/participant_features.csv")
df_visit = pd.read_csv("/work/MaleneJensen#0692/exam/output/visit_features.csv")
df_utts = pd.read_csv("/work/MaleneJensen#0692/exam/output/utterance_level_full.csv")

id_map = {
    old_id: f"P{str(i).zfill(2)}"
    for i, old_id in enumerate(df_part["ID"].unique())
}

def anonymize_ids(df, id_map):
    df = df.copy()
    df["ID"] = df["ID"].map(id_map)
    return df

df_utts = anonymize_ids(df_utts, id_map)
df_visit = anonymize_ids(df_visit, id_map)
df_part = anonymize_ids(df_part, id_map)

In [ ]:
#Remove identifying columns that were only used for debugging
for df in [df_utts]:
    df.drop(columns=["Filename", "Path"], inplace=True, errors="ignore")

In [ ]:
def age_to_months(age_str):
    if pd.isna(age_str):
        return np.nan
    m, d = age_str.split(";")
    return int(m) + int(d)/30.0

df_part["Age_months"] = df_part["Age"].apply(age_to_months)

In [ ]:
utterance_counts = (
    df_utts.groupby("ID")
    .size()
    .sort_values(ascending=True)
)

print(utterance_counts.head(10))

## Exclude participants with less than 30 utterances

In [ ]:
valid_ids = utterance_counts[utterance_counts >= 30].index

df_utts = df_utts[df_utts["ID"].isin(valid_ids)]
df_part = df_part[df_part["ID"].isin(valid_ids)]
df_visit = df_visit[df_visit["ID"].isin(valid_ids)]

## Visualize and analyze the effect of gender and age

In [ ]:
import matplotlib.pyplot as plt

df_visit_age = pd.read_csv("/work/MaleneJensen#0692/exam/df_visit_Age.csv")

mean_age_visit = (
    df_visit_age.groupby(["Group", "Visit"])["Age_months"]
    .mean()
    .reset_index()
)

plt.figure()
for group in mean_age_visit["Group"].unique():
    sub = mean_age_visit[mean_age_visit["Group"] == group]
    plt.plot(sub["Visit"], sub["Age_months"], marker="o", label=group)

plt.xlabel("Visit")
plt.ylabel("Mean age (months)")
plt.title("Mean age across visits by group")
plt.legend()
plt.show()


In [ ]:
gender_counts = (
    df_visit
    .drop_duplicates("ID")
    .groupby(["Group", "Gender"])
    .size()
    .unstack(fill_value=0)
)

gender_counts.plot(kind="bar")
plt.ylabel("Number of participants")
plt.title("Gender distribution by group")
plt.show()

## Embedding Classifier

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model again 
model = "all-mpnet-base-v2"

# Load in the embeddings
embeddings = np.load("output/sbert_embeddings.npy")
print(embeddings.shape)

In [ ]:
df_ut = pd.read_csv("/work/MaleneJensen#0692/exam/output/utterance_level_full.csv")
df_ut = df_utts = anonymize_ids(df_ut, id_map)

# Convert embeddings to dataframe
emb_cols = [f"SBERT_{i}" for i in range(embeddings.shape[1])]
df_emb = pd.DataFrame(embeddings, columns=emb_cols)


meta_cols = ["ID", "Age", "Gender", "Group", "Visit"]
for col in meta_cols:
    df_emb[col] = df_ut[col].values

df_emb = df_emb[df_emb["ID"].isin(valid_ids)]

In [ ]:
df_visit = df_visit.merge(
    df_part[["ID", "Age_months"]],
    on="ID",
    how="left"
)

df_visit["Age_months_visit"] = (
    df_visit["Age_months"] +
    (df_visit["Visit"] - 1) * 4
)

df_utts = df_utts.merge(
    df_visit[["ID", "Visit", "Age_months_visit"]],
    on=["ID", "Visit"],
    how="left"
)

df_age = df_part[["ID", "Age_months"]]

df_emb = df_emb.merge(
    df_age,
    on="ID",
    how="left"
)

In [ ]:
df_emb["Gender_num"] = df_emb["Gender"].map({
    "male": 0, "female": 1
})

X = df_emb.drop(
    columns=["ID", "Group", "Gender", "Age"]
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df_emb[emb_cols]
y = df_emb["Group"].map({"ASD": 1, "TD": 0})

# 80% train+val, 20% test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=88)
train_val_idx, test_idx = next(gss1.split(X, y, groups=df_emb["ID"]))

# From the 80%, split into 60% train / 20% val
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=88)
train_idx, val_idx = next(
    gss2.split(
        X.iloc[train_val_idx],
        y.iloc[train_val_idx],
        groups=df_emb.iloc[train_val_idx]["ID"]
    )
)

# Train
X_train = X.iloc[train_val_idx].iloc[train_idx]
y_train = y.iloc[train_val_idx].iloc[train_idx]

# Validation
X_val = X.iloc[train_val_idx].iloc[val_idx]
y_val = y.iloc[train_val_idx].iloc[val_idx]

# Test
X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

In [ ]:
#Check for participants in multiple groups
train_ids = set(df_emb.iloc[train_val_idx].iloc[train_idx]["ID"])
val_ids   = set(df_emb.iloc[train_val_idx].iloc[val_idx]["ID"])
test_ids  = set(df_emb.iloc[test_idx]["ID"])

print("Train–Val overlap:", train_ids & val_ids)
print("Train–Test overlap:", train_ids & test_ids)
print("Val–Test overlap:", val_ids & test_ids)


### Train the Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=88)

clf.fit(X_train, y_train)

In [ ]:
# Print classification report
from sklearn.metrics import classification_report

y_test_pred = clf.predict(X_test)
print("Test performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_test_pred)

# Normalize by true labels (rows)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2%",
    cmap="Blues",
    xticklabels=["TD", "ASD"],
    yticklabels=["TD", "ASD"]
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Normalized Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

### Visualizing Embedding Space

In [ ]:
df_utts_clean = df_utts.copy()

df_utts_clean["clean_utterance"] = df_utts_clean["clean_utterance"].astype(str)
df_utts_clean = df_utts_clean[df_utts_clean["clean_utterance"].str.strip() != ""]

print("Remaining utterances:", len(df_utts_clean))

# Get indices that still exist
valid_emb_indices = df_utts_clean["SBERT_emb_index"].dropna().astype(int)

# Subset embeddings
embeddings_clean = embeddings[valid_emb_indices.values]

In [ ]:
y = df_utts_clean["Group"].map({"TD": 0, "ASD": 1}).values

In [ ]:
# PCA
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=88)
X_pca = pca.fit_transform(embeddings_clean)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[y==0, 0], X_pca[y==0, 1], alpha=0.4, label="TD")
plt.scatter(X_pca[y==1, 0], X_pca[y==1, 1], alpha=0.4, label="ASD")
plt.legend()
plt.title("SBERT embedding space (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

In [ ]:
# UMAP
%pip install umap-learn
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=88)
X_umap = reducer.fit_transform(embeddings_clean)

plt.figure(figsize=(8,6))
plt.scatter(X_umap[y==0, 0], X_umap[y==0, 1], alpha=0.4, label="TD")
plt.scatter(X_umap[y==1, 0], X_umap[y==1, 1], alpha=0.4, label="ASD")
plt.legend()
plt.title("SBERT embedding space (UMAP)")
plt.tight_layout()
plt.show()

## Psycholinguistics Classifier

In [ ]:
# Figuring out what features to include in the dataframe
df_utts = pd.read_csv("/work/MaleneJensen#0692/exam/df_utts_AgePerVisit.csv")
df_utts = anonymize_ids(df_utts, id_map)


visit_features = df_visit[
    [
        "ID",
        "Visit",
        "MLU_visit",
        "IQR_LU_visit",
        "UniqueWords_visit",
    ]
]

In [ ]:
df_utts = df_utts.merge(
    visit_features,
    on=["ID", "Visit"],
    how="left"
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

df_utts["Gender_num"] = df_utts["Gender"].map({
    "male": 0, "female": 1})

features = [
    # Utterance-level
    "UniqueWords_utterance", "Disfluency_count", "Utterance_length", "PronounCount",

    # Visit-level
    "MLU_visit", "IQR_LU_visit", "UniqueWords_visit"
]

X = df_utts[features]
y = df_utts["Group"].map({"ASD": 1, "TD": 0})

# 80% train+val, 20% test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=88)
train_val_idx, test_idx = next(gss1.split(X, y, groups=df_utts["ID"]))

# From the 80%, split into 60% train / 20% val
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=88)
train_idx, val_idx = next(
    gss2.split(
        X.iloc[train_val_idx],
        y.iloc[train_val_idx],
        groups=df_utts.iloc[train_val_idx]["ID"]
    )
)

# Train
X_train = X.iloc[train_val_idx].iloc[train_idx]
y_train = y.iloc[train_val_idx].iloc[train_idx]

# Validation
X_val = X.iloc[train_val_idx].iloc[val_idx]
y_val = y.iloc[train_val_idx].iloc[val_idx]

# Test
X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]


In [ ]:
train_ids = set(df_utts.iloc[train_val_idx].iloc[train_idx]["ID"])
val_ids   = set(df_utts.iloc[train_val_idx].iloc[val_idx]["ID"])
test_ids  = set(df_utts.iloc[test_idx]["ID"])

print("Train–Val overlap:", train_ids & val_ids)
print("Train–Test overlap:", train_ids & test_ids)
print("Val–Test overlap:", val_ids & test_ids)

### Training Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=88
)

clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

y_test_pred = clf.predict(X_test)
print("Test performance:")
print(classification_report(y_test, y_test_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_test_pred)

# Normalize by true labels (rows)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2%",
    cmap="Blues",
    xticklabels=["TD", "ASD"],
    yticklabels=["TD", "ASD"]
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Normalized Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

### Visualizations

In [ ]:
# Correlation Matrix
import seaborn as sns
import matplotlib.pyplot as plt

feature_df = X_val.copy()
corr = feature_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Printing Importance 
import pandas as pd

importances = clf.feature_importances_
feat_imp = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

feat_imp.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(feat_imp["feature"][:10][::-1], feat_imp["importance"][:10][::-1])
plt.xlabel("Importance (MDI)")
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.lineplot(
    data=df_visit,
    x="Visit",
    y="UniqueWords_visit",
    hue="Group",
    estimator="mean",
    errorbar="se"
)
plt.title("Unique Words Over Time by Group")
plt.ylabel("Unique Words (avg)")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(
    data=df_visit,
    x="Visit",
    y="IQR_LU_visit",
    hue="Group",
    estimator="mean",
    errorbar="se"
)
plt.title("IQR of Utterance Length Over Time by Group")
plt.ylabel("IQR of Length of Utterance")
plt.tight_layout()
plt.show()

In [ ]:
sns.lineplot(
    data=df_visit,
    x="Visit",
    y="AvgPronounCount_visit",
    hue="Group",
    estimator="mean",
    errorbar="se"
)
plt.title("Pronoun Count Over Time by Group")
plt.ylabel("Pronoin Count (avg)")
plt.tight_layout()
plt.show()